# Akamai workshop · vLLM + token kinetics + FCO/FCG smoke test

**Run all cells in the existing Akamai JupyterLab Python kernel. No terminal commands or additional installation required.**

This single read-only experiment checks the workshop's existing vLLM endpoint and Kubernetes namespace, samples its native Prometheus metrics, runs **four small sequential Qwen3-4B completions**, calculates per-request tokens/s and **inter-request tokens/s²**, creates FCO/FCG-style evidence projections, and verifies arithmetic/identity from saved receipts.

**Evidence rules:** no mocked results; failed or unavailable measurements remain `UNKNOWN`. Stream chunks are **not** individual tokenizer tokens. tokens/s² here is variation in **whole-request throughput**, not token-by-token acceleration. Content hashing does not establish model accuracy, legal custody, a digital signature, or MMR commitment. This notebook neither mutates Kubernetes resources nor launches load tests.

Upstream workshop: [Akamai's original notebooks](https://github.com/akamai-developers/akamai-workshop-ai-inference). When available, import their `common.config` and `common.metrics` rather than duplicating infrastructure. Results stay on the Akamai server under `cell_akamai_results/`; inspect before publishing.

## 1 · Locate the existing workshop and configure the namespace

In [ ]:
import os, sys, json, hashlib, subprocess, importlib, math, uuid, time
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlsplit

import requests
import matplotlib.pyplot as plt
from openai import OpenAI

# User's kubeconfig already verified workshop-s05; the notebook kernel need not inherit terminal exports.
os.environ.setdefault("NAMESPACE", "workshop-s05")

# Prefer importing the already-installed workshop's helpers. The upload may be in a sibling directory.
WORKSHOP_ROOT_OVERRIDE = ""  # optional: absolute path only if automatic discovery fails
p = Path.cwd().resolve()
candidates = ([Path(WORKSHOP_ROOT_OVERRIDE)] if WORKSHOP_ROOT_OVERRIDE else []) + [
    p, *list(p.parents)[:3], Path.home(), Path.home() / "agents-that-own-their-inference",
]
for base in [p, p.parent, Path.home()]:
    try:
        candidates += [x for x in base.iterdir() if x.is_dir()][:35]
    except (OSError, PermissionError):
        pass
workshop_root = next((x.resolve() for x in candidates if (x / "common" / "config.py").is_file()
                      and (x / "common" / "metrics.py").is_file()), None)
if workshop_root:
    sys.path.insert(0, str(workshop_root))
    from common.config import get_settings, build_client, chat_extra
    from common.metrics import fetch_metrics_text, parse_metrics, parse_histogram_avg
    settings = get_settings()
    BASE_URL = settings.vllm_host
    MODEL_NAME = settings.model_name
    NAMESPACE = settings.namespace
    METRICS_URL = settings.metrics_url
    client = build_client(settings)
    print("Workshop helpers: FOUND", workshop_root)
else:
    BASE_URL = os.environ.get("VLLM_HOST", "http://vllm:8000/v1")
    MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen/Qwen3-4B")
    NAMESPACE = os.environ["NAMESPACE"]
    METRICS_URL = BASE_URL.rstrip("/").removesuffix("/v1") + "/metrics"
    client = OpenAI(base_url=BASE_URL, api_key=os.environ.get("VLLM_API_KEY", "not-needed"), timeout=90)
    print("Workshop helpers: NOT_FOUND (using OpenAI/requests read-only fallback)")

# Credentials are intentionally NOT printed, saved or hashed into public receipts.
print("Namespace:", NAMESPACE)
print("Model:", MODEL_NAME)
print("vLLM:", BASE_URL)
print("Metrics:", METRICS_URL)

## 2 · Module-0-equivalent preflight (read-only)

In [ ]:
preflight = {"model_list": "NOT_TESTED", "kubernetes": "NOT_TESTED", "metrics": "NOT_TESTED"}
resp = requests.get(BASE_URL.rstrip("/") + "/models", timeout=15)
resp.raise_for_status()
model_ids = [x.get("id") for x in resp.json().get("data", [])]
assert MODEL_NAME in model_ids, f"Configured {MODEL_NAME!r} absent from server model list: {model_ids}"
preflight["model_list"] = "PASS"
print("vLLM model list: PASS", model_ids)

try:
    check = subprocess.run(["kubectl", "-n", NAMESPACE, "get", "pods", "-o", "json"],
                           capture_output=True, text=True, timeout=12)
    if check.returncode == 0:
        pods = json.loads(check.stdout).get("items", [])
        pod_states = [{"name": x["metadata"]["name"],
                       "phase": x.get("status", {}).get("phase", "UNKNOWN")}
                      for x in pods]
        preflight["kubernetes"] = "PASS"
        print("Kubernetes read-only check: PASS", pod_states)
    else:
        pod_states = []
        preflight["kubernetes"] = "UNAVAILABLE"
        print("Kubernetes check unavailable:", check.stderr[:240])
except (OSError, subprocess.TimeoutExpired, ValueError) as err:
    pod_states = []
    preflight["kubernetes"] = "UNAVAILABLE"
    print("Kubernetes check unavailable:", str(err)[:240])

## 3 · Native vLLM gauges and histograms
The workshop helper `common.metrics` parses Prometheus exposition text. Gauge and histogram availability varies by vLLM version. A missing metric stays `None`—never zero-filled.

In [ ]:
def get_metrics_text():
    if workshop_root:
        return fetch_metrics_text(METRICS_URL, timeout=8)
    r = requests.get(METRICS_URL, timeout=8)
    r.raise_for_status()
    return r.text

def metrics_from_text(raw):
    if not raw:
        return {}
    if workshop_root:
        return parse_metrics(raw)
    # Standard Prometheus plaintext fallback: aggregate sample values by metric name.
    metrics = {}
    for line in raw.splitlines():
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        if len(parts) < 2:
            continue
        key = parts[0].split("{", 1)[0]
        try:
            value = float(parts[1])
            if math.isfinite(value):
                metrics[key] = metrics.get(key, 0.0) + value
        except ValueError:
            continue
    return metrics

def metric_value(m, name, alternate=None):
    v = m.get(name)
    return v if v is not None else m.get(alternate) if alternate else None

def observed_metrics(m):
    kv = metric_value(m, "vllm:kv_cache_usage_perc", "vllm:gpu_cache_usage_perc")
    return {
        "kv_cache_fraction": kv,
        "running_requests": m.get("vllm:num_requests_running"),
        "waiting_requests": m.get("vllm:num_requests_waiting"),
        "generation_tokens_total": m.get("vllm:generation_tokens_total"),
        "ttft_cumulative_mean_s": (m.get("vllm:time_to_first_token_seconds_sum") / m["vllm:time_to_first_token_seconds_count"]
          if m.get("vllm:time_to_first_token_seconds_count", 0) > 0 else None),
        "tpot_cumulative_mean_s": (m.get("vllm:time_per_output_token_seconds_sum") / m["vllm:time_per_output_token_seconds_count"]
          if m.get("vllm:time_per_output_token_seconds_count", 0) > 0 else None),
    }

try:
    initial_metrics_raw = get_metrics_text()
    initial_metrics = metrics_from_text(initial_metrics_raw)
    preflight["metrics"] = "PASS"
except (requests.RequestException, TimeoutError, OSError) as err:
    initial_metrics_raw, initial_metrics = None, {}
    preflight["metrics"] = "UNAVAILABLE"
    print("Metric endpoint unavailable:", repr(err))

print("Metrics status:", preflight["metrics"])
print("Initial gauges / cumulative histograms:")
print(json.dumps(observed_metrics(initial_metrics), indent=2))

## 4 · Freeze the tiny experiment
Four serial completions of one prompt, temperature 0, bounded output budget. No stress test, model swap, kube changes or training. Stream usage is taken from the server, **not inferred from chunk count**.

In [ ]:
EXPERIMENT_ID = "EXP-AKAMAI-TOKEN-001"
RUNS = 4
MAX_TOKENS = 96
PROMPT = "Explain one measurable benefit and one limit of KV caching in two sentences."
RUN_ID = "akm-" + uuid.uuid4().hex[:12]

def canonical_bytes(data):
    return json.dumps(data, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                      allow_nan=False).encode("utf-8")

def digest(data):
    return "sha256:" + hashlib.sha256(data).hexdigest()

def content_id(data):
    return digest(canonical_bytes(data))

# Local math self-test with explicitly synthetic inputs; it is not a server measurement.
test_a = {"duration_ns": 2_000_000_000, "completion_tokens": 20}
test_b = {"duration_ns": 1_000_000_000, "completion_tokens": 20}
def throughput(row):
    tokens = row.get("completion_tokens")
    duration_ns = row.get("duration_ns")
    return (tokens * 1e9 / duration_ns
            if isinstance(tokens, int) and duration_ns and duration_ns > 0 else None)
assert throughput(test_a) == 10.0 and throughput(test_b) == 20.0
assert content_id({"x": 1, "y": 2}) == content_id({"y": 2, "x": 1})
print("Deterministic math / canonical identity self-test: PASS (synthetic inputs only)")
print("Experiment ID:", EXPERIMENT_ID, "Run:", RUN_ID)

## 5 · Execute four real GPU-backed requests
Each request saves timing, streamed chunk timing (not token timing), vLLM metrics snapshots and final completion-token usage. The metrics capture occurs **outside** the timed request window. No prompts, generated text or credentials are exported.

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def safe_metrics():
    try:
        raw = get_metrics_text()
        return {"status": "OBSERVED", "raw": raw, "parsed": observed_metrics(metrics_from_text(raw)),
                "raw_sha256": digest(raw.encode("utf-8"))}
    except Exception as e:
        return {"status": "UNAVAILABLE", "raw": None, "parsed": {},
                "raw_sha256": None, "error_type": type(e).__name__}

records = []
metrics_snapshots = []
for i in range(RUNS):
    before = safe_metrics()
    metrics_snapshots.append({"run": i + 1, "phase": "before", **before})
    start_ns = time.perf_counter_ns()
    started_at = utc_now()
    first_output_ns = None
    chunk_observations = []
    completion_tokens = None
    response_id = None
    response_text_sha = None
    status, error_type = "OBSERVED", None
    text_chunks = []
    end_ns = None
    try:
        kwargs = dict(model=MODEL_NAME,
                      messages=[{"role": "user", "content": PROMPT}],
                      temperature=0, max_tokens=MAX_TOKENS,
                      stream=True, stream_options={"include_usage": True})
        if not workshop_root and "qwen3" in MODEL_NAME.lower():
            kwargs["extra_body"] = {"chat_template_kwargs": {"enable_thinking": False}}
        stream = client.chat.completions.create(**kwargs)
        for event in stream:
            arrival_ns = time.perf_counter_ns()
            response_id = response_id or getattr(event, "id", None)
            usage = getattr(event, "usage", None)
            if usage and usage.completion_tokens is not None:
                completion_tokens = int(usage.completion_tokens)
            for choice in (event.choices or []):
                delta = choice.delta
                chunk = (getattr(delta, "content", None) or "") + (getattr(delta, "reasoning_content", None) or "")
                if chunk:
                    if first_output_ns is None:
                        first_output_ns = arrival_ns
                    text_chunks.append(chunk)
                    chunk_observations.append({"elapsed_ns": arrival_ns - start_ns,
                                               "chunk_content_sha256": digest(chunk.encode("utf-8"))})
        end_ns = time.perf_counter_ns()
        response_text_sha = digest("".join(text_chunks).encode("utf-8"))
    except Exception as e:
        end_ns = time.perf_counter_ns()
        status, error_type = "FAILED", type(e).__name__
    after = safe_metrics()
    metrics_snapshots.append({"run": i + 1, "phase": "after", **after})
    row = {"run": i+1, "status": status, "error_type": error_type,
           "model": MODEL_NAME, "namespace": NAMESPACE, "response_id": response_id,
           "started_utc": started_at, "start_ns": start_ns,
           "duration_ns": end_ns - start_ns,
           "first_nonempty_chunk_elapsed_ns": (first_output_ns-start_ns if first_output_ns else None),
           "completion_tokens": completion_tokens,
           "stream_chunk_count": len(chunk_observations),
           "chunks": chunk_observations, "response_sha256": response_text_sha,
           "prompt_sha256": digest(PROMPT.encode("utf-8")),
           "before_metric_sha256": before["raw_sha256"],
           "after_metric_sha256": after["raw_sha256"]}
    records.append(row)
    print(f"Request {i+1}: status={status}, server tokens={completion_tokens}, "
          f"duration={(end_ns-start_ns)/1e9:.2f} s, chunks={len(chunk_observations)}")

successful = [r for r in records if r["status"] == "OBSERVED"]
print(f"Real completions: {len(successful)}/{RUNS}")
if len(successful) == 0:
    raise RuntimeError("No real vLLM completions observed. Inspect endpoint and request errors; no synthetic data substituted.")

## 6 · Derived token kinetics
Request throughput \(v_i=N_i/T_i\) uses the **reported completion-token count** and whole-request wall duration. For request interval midpoint \(m_i\), \(a_i=(v_i-v_{i-1})/(m_i-m_{i-1})\) has units tokens/s². A request that fails or lacks token usage yields `None`, not a fabricated count. Timed first nonempty chunk is an approximate first-output latency, **not guaranteed TTFT**.

In [ ]:
def derive(rows):
    # Deterministic pure function over preserved raw records.
    output = []
    for row in rows:
        v = throughput(row) if row.get("status") == "OBSERVED" else None
        midpoint_ns = row["start_ns"] + row["duration_ns"] / 2
        a = None
        if output:
            previous = output[-1]
            delta_t = (midpoint_ns - previous["midpoint_ns"]) / 1e9
            if v is not None and previous["tokens_per_s"] is not None and delta_t > 0:
                a = (v - previous["tokens_per_s"]) / delta_t
        output.append({"run": row["run"], "status": row["status"],
                       "midpoint_ns": midpoint_ns,
                       "tokens_per_s": v, "inter_request_tokens_per_s2": a,
                       "first_nonempty_chunk_ms": (row["first_nonempty_chunk_elapsed_ns"] / 1e6
                           if row["first_nonempty_chunk_elapsed_ns"] is not None else None)})
    return output

derived = derive(records)
for r in derived:
    print(f"#{r['run']}: tok/s={r['tokens_per_s']}, tok/s²={r['inter_request_tokens_per_s2']}, "
          f"first nonempty chunk={r['first_nonempty_chunk_ms']} ms")
assert len(derived) == RUNS
print("Arithmetic recomputation from in-memory raw record: PASS")

## 7 · Plot observed request throughput and acceleration
Missing data stays missing. These are separate sequential requests; no claim is made about instantaneous per-token acceleration or saturation.

In [ ]:
x = [r["run"] for r in derived]
for key, ylabel, title in [
    ("tokens_per_s", "Completion tokens/s", "Observed request-level output throughput"),
    ("inter_request_tokens_per_s2", "Tokens/s²", "Observed inter-request throughput acceleration")
]:
    plt.figure(figsize=(7.5, 3.5))
    y = [float("nan") if r[key] is None else r[key] for r in derived]
    plt.plot(x, y, "o-")
    if key.endswith("s2"):
        plt.axhline(0, linestyle="--", linewidth=1)
    plt.xticks(x)
    plt.xlabel("Sequential request")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

print("Native vLLM cache-gauge samples (fraction, not percent):")
for s in metrics_snapshots:
    print(s["run"], s["phase"], s["parsed"].get("kv_cache_fraction"))

## 8 · Addressable FCO/FCG projection and on-server receipts

Create distinct content IDs (canonical SHA-256), occurrence IDs (request/observation context), and **declared** typed relationships. These are local FCO/FCG projections; they are **not** signed, independently attributed, MMR-committed, or scientific quality evidence. The original raw vLLM Prometheus text is preserved with checksums. No request prompt text, generated responses, model credentials, or kubeconfig exported.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out_dir = Path.cwd() / "cell_akamai_results" / f"{EXPERIMENT_ID}_{stamp}_{RUN_ID}"
out_dir.mkdir(parents=True, exist_ok=False)

fcos, edges = [], []
def add_fco(kind, occurrence_id, payload):
    obj = {"kind": kind, "occurrence_id": occurrence_id,
           "content_id": content_id({"kind": kind, "payload": payload}), "payload": payload}
    fcos.append(obj)
    return obj["content_id"]

source_id = add_fco("SourceEnvironment", f"{RUN_ID}:environment",
                    {"model": MODEL_NAME, "namespace": NAMESPACE,
                     "endpoint_host": urlsplit(BASE_URL).hostname,
                     "model_revision": "UNKNOWN", "tokenizer_revision": "UNKNOWN",
                     "preflight": preflight})
for row, kinetic in zip(records, derived):
    rid = f"{RUN_ID}:request:{row['run']}"
    obs_id = add_fco("InferenceObservation", rid, row)
    kin_id = add_fco("DerivedTokenKinetics", rid+":derived", kinetic)
    edges.extend([
        {"source": obs_id, "target": source_id, "type": "OBSERVED_IN"},
        {"source": kin_id, "target": obs_id, "type": "DERIVED_FROM"}
    ])

for snapshot in metrics_snapshots:
    raw = snapshot["raw"]
    if raw is not None:
        raw_name = f"metrics_run{snapshot['run']}_{snapshot['phase']}.prom"
        (out_dir / raw_name).write_text(raw, encoding="utf-8")
        assert digest((out_dir / raw_name).read_bytes()) == snapshot["raw_sha256"]

with (out_dir / "fco_atoms.jsonl").open("w", encoding="utf-8") as fp:
    for atom in fcos:
        fp.write(json.dumps(atom, sort_keys=True, allow_nan=False)+"\n")
with (out_dir / "fcg_edges.jsonl").open("w", encoding="utf-8") as fp:
    for edge in edges:
        fp.write(json.dumps(edge, sort_keys=True)+"\n")

receipt = {
    "experiment_id": EXPERIMENT_ID, "run_id": RUN_ID,
    "source": {"workshop": "https://github.com/akamai-developers/akamai-workshop-ai-inference",
               "model": MODEL_NAME, "model_revision": "UNKNOWN",
               "namespace": NAMESPACE, "metrics_source": METRICS_URL},
    "observation_status": "OBSERVED" if successful else "NOT_TESTED",
    "measurement_scope": "sequential request throughput; not individual token dynamics",
    "unit_contract": {"tokens_per_s": "completion tokens / whole-request seconds",
                      "inter_request_tokens_per_s2": "difference in request tok/s / midpoint-separation seconds",
                      "first_nonempty_chunk_ms": "milliseconds; proxy for TTFT, not true token timestamp",
                      "kv_cache_fraction": "dimensionless fraction (0..1), not percent"},
    "preflight": preflight, "records": records, "derived": derived,
    "metrics_files": [{"run": s["run"], "phase": s["phase"],
                       "status": s["status"], "sha256": s["raw_sha256"],
                       "parsed": s["parsed"]} for s in metrics_snapshots],
    "signed": False, "mmr": "NOT_COMMITTED", "independent_reexecution": "NOT_TESTED",
    "anticube": "NOT_COMPUTED", "delta_g_star": "NOT_COMPUTED",
    "claim_state": "MEASURED_ONLY; NO_CAUSAL_OR_SCIENTIFIC_CLAIM"
}
receipt_file = out_dir / "experiment_receipt.json"
receipt_file.write_text(json.dumps(receipt, indent=2, sort_keys=True, allow_nan=False), encoding="utf-8")

# Verify from disk: recompute derived arithmetic, atom content IDs, edge referential integrity and metrics hashes.
loaded = json.loads(receipt_file.read_text(encoding="utf-8"))
assert loaded["derived"] == derive(loaded["records"])
loaded_atoms = [json.loads(line) for line in (out_dir / "fco_atoms.jsonl").read_text().splitlines()]
loaded_edges = [json.loads(line) for line in (out_dir / "fcg_edges.jsonl").read_text().splitlines()]
assert all(a["content_id"] == content_id({"kind": a["kind"], "payload": a["payload"]}) for a in loaded_atoms)
ids = {a["content_id"] for a in loaded_atoms}
assert all(e["source"] in ids and e["target"] in ids for e in loaded_edges)
for s in metrics_snapshots:
    if s["raw_sha256"]:
        assert digest((out_dir / f"metrics_run{s['run']}_{s['phase']}.prom").read_bytes()) == s["raw_sha256"]

verification = {"same_host_replay": "PASS", "checks": ["arithmetic", "content_IDs", "FCG_edge_endpoints", "raw_metrics_file_hashes"],
                "independent_host_reexecution": "NOT_TESTED", "mmr": "NOT_COMMITTED", "signed": False}
(out_dir / "verification.json").write_text(json.dumps(verification, indent=2), encoding="utf-8")
print("Receipt directory:", out_dir)
print("Local deterministic replay:", verification["same_host_replay"])
print("Source observations:", len(records), "Addressable atoms:", len(fcos), "typed relationships:", len(edges))
print("Independent host re-execution: NOT_TESTED; MMR: NOT_COMMITTED; signatures: NOT_SIGNED")

## Next steps

1. Share the printed metrics and final receipt **status**, not your credentials or raw kubeconfig.
2. If the notebook succeeds, compare its readings against the workshop's Module 3 and Module 7 plots. Avoid module 7's high-concurrency stress test until that is explicitly approved.
3. Feed **reviewed** Lane A measurements and separately admitted Lane B Genesis/Vita DAISY evidence to the model-comparison dashboard. This notebook intentionally does not calculate Anticube or ΔG*; those require validated scientific contracts and actual experiments.